# Phase 7 (Prior Case Retrieval) — c-3 hierarchical (hourglass) encoder-decoder + QLoRA

The c-3 **encoder** is used as a bi-encoder: query and candidate cases are cut into token windows, encoded, pooled to window vectors and compared (cosine / dot / maxsim). Training is contrastive (InfoNCE or margin) with in-batch and/or hard negatives. Output: top 7 candidate ids per test query (`prediction.json`), scored by Micro-F1@7.

QLoRA: chosen encoder linears stored in 4 bit (bitsandbytes), your LoRALinear `W + (alpha/r)BA` on top is the only thing that trains (additive LoRA, not LORMA).

**Phase 7 technique list:** full FT, partial/layer-wise FT, QLoRA, LORMA, prefix, prompt, IA3, KD. QLoRA is on it.

Run every cell top to bottom on a GPU. Everything you choose is in section **[H]** of Cell E1 (all left `None`).

## Cell 0 — Phase 1 imports, tokenizer class, model classes, HF login

In [1]:
# --- Phase 1 c-3 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


In [2]:
# --- Phase 1 c-3 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


In [3]:
# --- Phase 1 c-3 notebook, cell 20 (verbatim) ---
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# --- Phase 1 c-3 notebook, cell 22 (verbatim) ---
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    shorten_factor: int = 4
    n_pre_layers: int = 1
    n_bottleneck_layers: int = 4
    n_post_layers: int = 1

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x

# ---------------------------------------------------------
# 3. Hierarchical Resampling Operations
# ---------------------------------------------------------
class Downsample(nn.Module):
    def __init__(self, factor: int):
        super().__init__()
        self.factor = factor

    def forward(self, x: torch.Tensor):
        B, L, D = x.shape
        x = x.view(B, L // self.factor, self.factor, D)
        return x.mean(dim=2)


class Upsample(nn.Module):
    def __init__(self, factor: int):
        super().__init__()
        self.factor = factor

    def forward(self, x: torch.Tensor):
        return x.repeat_interleave(self.factor, dim=1)


def shift_right(x: torch.Tensor, shift: int):
    if shift == 0:
        return x
    shifted = F.pad(x, (0, 0, shift, 0), value=0.0)
    return shifted[:, :-shift, :]

# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class HourglassEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        
        # Positional Encoding (using standard learned embedding for simplicity, 
        # replace with sinusoidal if strictly required)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        # ENCODER
        self.encoder_pre = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_pre_layers)])
        self.encoder_hourglass = EncoderHourglassBlock(cfg, cfg.shorten_factor, cfg.n_bottleneck_layers)
        self.encoder_post = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_post_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        # DECODER
        self.decoder_pre = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_pre_layers)])
        self.decoder_hourglass = DecoderHourglassBlock(cfg, cfg.shorten_factor, cfg.n_bottleneck_layers)
        self.decoder_post = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_post_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)
        
        # Output Head
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight # Weight tying

    def get_positional_embeddings(self, seq_len: int, device: torch.device):
        positions = torch.arange(seq_len, device=device).unsqueeze(0)
        return self.pos_emb(positions)

    def forward(self, enc_idx: torch.Tensor, dec_idx: torch.Tensor) -> torch.Tensor:
        B_enc, T_enc = enc_idx.shape
        B_dec, T_dec = dec_idx.shape
        
        # 1. ENCODER PASS
        enc_x = self.tok_emb(enc_idx) + self.get_positional_embeddings(T_enc, enc_idx.device)
        for block in self.encoder_pre:
            enc_x = block(enc_x)
        enc_x = self.encoder_hourglass(enc_x)
        for block in self.encoder_post:
            enc_x = block(enc_x)
        encoder_out = self.encoder_ln(enc_x)

        # 2. DECODER PASS
        dec_x = self.tok_emb(dec_idx) + self.get_positional_embeddings(T_dec, dec_idx.device)
        for block in self.decoder_pre:
            dec_x = block(dec_x, encoder_out)
        dec_x = self.decoder_hourglass(dec_x, encoder_out)
        for block in self.decoder_post:
            dec_x = block(dec_x, encoder_out)
            
        dec_x = self.decoder_ln(dec_x)
        logits = self.lm_head(dec_x)
        
        return logits


# --- Phase 1 c-3 notebook, cell 23 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    """Wrapper that keeps one forward(enc_idx, dec_idx) -> logits (B, T_dec, V) and pads the DECODER input on
    the right to a multiple of shorten_factor (safe: the decoder is causal, padding cannot change earlier logits).
    The ENCODER input is bidirectional, so it must already be a multiple of shorten_factor (never padded)."""
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = HourglassEncoderDecoder(cfg)          # YOUR model

    def forward(self, enc_idx, dec_idx):
        f = self.cfg.shorten_factor
        assert enc_idx.size(1) % f == 0, "encoder length must be a multiple of shorten_factor"
        T = dec_idx.size(1)
        pad = (-T) % f
        if pad:
            dec_idx = F.pad(dec_idx, (0, pad), value=0)    # id 0 = <pad>
        logits = self.core(enc_idx, dec_idx)
        return logits[:, :T, :]


In [4]:
# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell A — find and load the Phase 1 c-3 model and tokenizer (fix ROOT to what the find command shows)

In [5]:
!find /kaggle/input -path "*phase-1-c-3*" \( -name "*.pt" -o -name "*.pkl" \) -exec ls -la {} \;


-rw-rw-rw- 1 root root 476977 Nov  3 05:53 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/bpe_tokenizer.pkl
-rw-rw-rw- 1 root root 476977 Nov  3 05:53 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl
-rw-rw-rw- 1 root root 220615443 Nov  3 05:53 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/model_encdec/model_final.pt


In [6]:
# =====================================================================================
# CELL A — load the Phase 1 c-3 hourglass encoder-decoder and its tokenizer
# =====================================================================================
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, max_seq_len 1536, shorten 4
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full"   # fix to what ls showed
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-3's own tokenizer (vocab 11022)
ck = torch.load(f"{ROOT}/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| max_seq_len", cfg.max_seq_len, "| shorten", cfg.shorten_factor)


<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | max_seq_len 1536 | shorten 4


## Cell B — encoder sanity check

In [7]:
# =====================================================================================
# CELL B — per-token vectors from the c-3 ENCODER + sanity check (expect max diff 0.0)
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor; T must be a multiple of 4 and <= 1536
    c = lm.core
    T = idx.size(1)
    assert T % c.cfg.shorten_factor == 0 and T <= c.cfg.max_seq_len
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_pre:
        x = blk(x)
    x = c.encoder_hourglass(x)            # 4 bottleneck layers on the 4x shortened sequence, upsampled back
    for blk in c.encoder_post:
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the encoder output inside the full model's forward
got = {}
h = lm.core.encoder_ln.register_forward_hook(lambda m, i, o: got.__setitem__("e", o))
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    lm(idx, torch.full((1, 4), 2, device=device)); f = features(idx)
h.remove(); print("max diff", (f - got["e"]).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)


max diff 0.0 (1, 64, 560)


## Cell C — Phase 7 data: queries + candidate pools per split

In [8]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 7 (Prior Case Retrieval) data: queries + candidate pools per split
#   Train/train.json + train_candidates.json, Dev/dev.json + dev_candidates.json,
#   Test/test.json + test_candidates.json. Each split's queries are ranked ONLY against
#   that split's own candidate pool. "text" = the case split into sentences (joined here).
# The exact JSON layout is not in the spec, so the reader accepts the usual layouts and
# prints what it found. Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, random
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P7_DIR = "Phase_7_PCR"
print("Phase 7 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_7" in f])

def get7(name):
    """Reads Phase_7_PCR/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P7_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

ID_FIELDS   = ("id", "query_id", "candidate_id", "case_id", "qid", "cid", "doc_id")
TEXT_FIELDS = ("text", "document", "case_text", "content", "judgment", "sentences")
REL_FIELDS  = ("relevant_candidates", "relevant_cases", "relevant", "positives", "gold", "labels")
CAND_FIELDS = ("candidates", "candidate_ids", "candidate_pool")   # only if a query lists its own pool

def norm_id(x):
    """IDs as strings. Integer IDs are zero-padded to 10 digits like the spec's example output."""
    if isinstance(x, (int, np.integer)):
        return f"{int(x):010d}"
    return str(x).strip()

def join_text(t):
    if isinstance(t, dict):                              # e.g. {"sentences": [...]}
        t = next((t[f] for f in TEXT_FIELDS if f in t), list(t.values()))
    if isinstance(t, (list, tuple)):
        return " ".join(str(s).strip() for s in t if str(s).strip())
    return str(t)

def _unwrap(raw):
    # {"data": [...]} / {"queries": {...}} -> inner; never unwrap {"<case id>": {...}}
    while (isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict))
           and not norm_id(next(iter(raw))).isdigit()):
        raw = next(iter(raw.values()))
    idf = next((f for f in ID_FIELDS if isinstance(raw, dict) and isinstance(raw.get(f), list)), None)
    if idf:                                              # columns layout {"id": [...], "text": [...], ...}
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw[idf]))]
    return raw

def as_cases(raw, name, is_query):
    """-> {case_id: {"text": str, "rel": [ids] or None, "cands": [ids] or None}} in file order."""
    raw = _unwrap(raw)
    items = list(raw.items()) if isinstance(raw, dict) else [(None, d) for d in raw]
    out, n_dup = {}, 0
    for k, d in items:
        if not isinstance(d, dict):                      # {id: "text"} or {id: [sentences]}
            d = {"text": d}
        idf = next((f for f in ID_FIELDS if f in d), None)
        cid = norm_id(d[idf] if idf else k)
        assert cid not in (None, "None"), f"{name}: record without an id (keys {list(d)})"
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"{name} {cid}: no text field (keys {list(d)})"
        rf = next((f for f in REL_FIELDS if f in d), None) if is_query else None
        cf = next((f for f in CAND_FIELDS if f in d), None) if is_query else None
        rec = {"text": join_text(d[tf]),
               "rel": [norm_id(x) for x in d[rf]] if rf and d[rf] is not None else None,
               "cands": [norm_id(x) for x in d[cf]] if cf and d[cf] is not None else None}
        if cid in out:
            n_dup += 1
            if out[cid]["text"] != rec["text"]:
                print(f"WARNING {name}: id {cid} repeated with different text, keeping the first")
            continue
        out[cid] = rec
    print(f"{name}: {len(items)} records -> {len(out)} cases (repeated ids skipped {n_dup})")
    return out

def _peek(raw, name):
    r = _unwrap(raw)
    first = r[0] if isinstance(r, list) else next(iter(r.values()))
    print(f"{name} layout: {type(r).__name__} of {len(r)} | first record:",
          list(first) if isinstance(first, dict) else type(first).__name__)

queries, pools = {}, {}
for split, folder in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
    _q = get7(f"{folder}/{split}.json"); _peek(_q, f"{split}.json")
    _c = get7(f"{folder}/{split}_candidates.json"); _peek(_c, f"{split}_candidates.json")
    queries[split] = as_cases(_q, f"{split} queries", True)
    pools[split] = as_cases(_c, f"{split} candidates", False)
del _q, _c

# test labels are private: ignore anything that looks like them
for d in queries["test"].values():
    d["rel"] = None

# Gold sets (train/dev): drop relevant ids that are not in the split's pool (cannot be retrieved)
GOLD = {}
for split in ("train", "dev"):
    pool, g, n_missing, n_rel = pools[split], {}, 0, 0
    for q, d in queries[split].items():
        rel = list(dict.fromkeys(d["rel"] or []))
        n_rel += len(rel)
        keep = [c for c in rel if c in pool and c != q]
        n_missing += len(rel) - len(keep)
        g[q] = set(keep)
    GOLD[split] = g
    n_empty = sum(not s for s in g.values())
    print(f"{split}: {n_rel} relevant links, {n_missing} not in the {split} pool (dropped) | "
          f"queries with no retrievable relevant case: {n_empty}")

# Queries with no gold cannot teach anything; they still count in the dev score (as the evaluator would)
TRAIN_QIDS = [q for q in queries["train"] if GOLD["train"][q]]

for split in ("train", "dev", "test"):
    Q, P = queries[split], pools[split]
    qw = np.array([len(d["text"].split()) for d in Q.values()])
    cw = np.array([len(d["text"].split()) for d in P.values()])
    line = (f"{split:5s} queries {len(Q):5d} | pool {len(P):6d} | query words median {np.median(qw):.0f} "
            f"p90 {np.percentile(qw, 90):.0f} | candidate words median {np.median(cw):.0f} "
            f"p90 {np.percentile(cw, 90):.0f}")
    if split in GOLD:
        gs = np.array([len(s) for s in GOLD[split].values()])
        line += f" | gold per query mean {gs.mean():.2f} median {np.median(gs):.0f} max {gs.max()} | >7: {(gs > 7).mean():.1%}"
    n_self = sum(q in P for q in Q)
    if n_self:
        line += f" | {n_self} queries are also in their own pool (excluded when ranking)"
    if any(d["cands"] for d in Q.values()):
        line += " | queries carry their own candidate lists (ranking restricted to them)"
    print(line)
print("pool overlap train∩dev", len(set(pools["train"]) & set(pools["dev"])),
      "| train∩test", len(set(pools["train"]) & set(pools["test"])),
      "| dev∩test", len(set(pools["dev"]) & set(pools["test"])))
print("first test query ids:", list(queries["test"])[:3])


Phase 7 files on HF: ['Phase_7_PCR/Dev/dev.json', 'Phase_7_PCR/Dev/dev_candidates.json', 'Phase_7_PCR/Test/test.json', 'Phase_7_PCR/Test/test_candidates.json', 'Phase_7_PCR/Train/train.json', 'Phase_7_PCR/Train/train_candidates.json']


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train.json layout: dict of 827 | first record: ['text', 'relevant_candidates']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

... (7 outputs left out)

test.json layout: dict of 237 | first record: ['text']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test_candidates.json layout: dict of 1727 | first record: ['text']
test queries: 237 records -> 237 cases (repeated ids skipped 0)
test candidates: 1727 records -> 1727 cases (repeated ids skipped 0)
train: 5308 relevant links, 0 not in the train pool (dropped) | queries with no retrievable relevant case: 10
dev: 973 relevant links, 0 not in the dev pool (dropped) | queries with no retrievable relevant case: 0
train queries   827 | pool   4320 | query words median 4508 p90 12602 | candidate words median 3632 p90 12422 | gold per query mean 6.42 median 4 max 52 | >7: 30.0% | 827 queries are also in their own pool (excluded when ranking)
dev   queries   118 | pool   1023 | query words median 4620 p90 15333 | candidate words median 4036 p90 16599 | gold per query mean 8.25 median 5 max 92 | >7: 33.1% | 118 queries are also in their own pool (excluded when ranking)
test  queries   237 | pool   1727 | query words median 4653 p90 14065 | candidate words median 3947 p90 13491 | 237 queries ar

## Cell D — tokenisation (cached), Micro-F1@7 scorer, prediction.json writer, BM25 baseline

In [9]:
# =====================================================================================
# CELL D — [ANCILLARY] tokenisation (cached), Micro-F1@7 scorer, prediction.json writer,
#          BM25 baseline (also used as a hard-negative source in Cell E2 if you choose it)
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import os, re, json, glob, time, zipfile, pickle, hashlib
import numpy as np
import scipy.sparse as sp
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
TOPK = 7                                 # the evaluator scores exactly the first 7 ids per query

# -------------------------------------------------------------------------------------
# Tokenise every query and candidate once with the c-3 BPE tokenizer -> d["ids"].
# The Python BPE is slow on long judgments, so: identical texts are encoded once, all CPU cores
# are used (fork), and the result is saved to /kaggle/working/p7_c3_tok.pkl. Attach a finished
# run's output as input next time and the cache is found and reused automatically.
# -------------------------------------------------------------------------------------
TOK_CACHE_OUT = "/kaggle/working/p7_c3_tok.pkl"

def _h(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def _enc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

_all_docs = [d for s in ("train", "dev", "test") for part in (queries[s], pools[s]) for d in part.values()]
_cache = {}
for _p in sorted(glob.glob("/kaggle/input/**/p7_c3_tok.pkl", recursive=True)) + [TOK_CACHE_OUT]:
    if os.path.exists(_p):
        _cache.update(pickle.load(open(_p, "rb")))
        print(f"token cache loaded from {_p}: {len(_cache)} texts")
_todo = list({_h(d["text"]): d["text"] for d in _all_docs if _h(d["text"]) not in _cache}.items())
print(f"{len(_all_docs)} documents, {len(_todo)} distinct texts still to tokenise")
if _todo:
    t0 = time.time()
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(os.cpu_count()) as _pool:
            _ids = list(tqdm(_pool.imap(_enc, [t for _, t in _todo], chunksize=4), total=len(_todo), desc="tokenising"))
    except Exception as e:                                       # no fork available: one process
        print("parallel tokenising failed, falling back to one process:", repr(e))
        _ids = [_enc(t) for _, t in tqdm(_todo, desc="tokenising")]
    _cache.update({h: ids for (h, _), ids in zip(_todo, _ids)})
    pickle.dump(_cache, open(TOK_CACHE_OUT, "wb"))
    print(f"tokenised in {(time.time() - t0) / 60:.1f} min, cache saved to {TOK_CACHE_OUT}")
for d in _all_docs:
    d["ids"] = _cache[_h(d["text"])]
del _cache, _todo

for s in ("train", "dev", "test"):
    for kind, part in (("queries", queries[s]), ("pool", pools[s])):
        L = np.array([len(d["ids"]) for d in part.values()])
        print(f"{s:5s} {kind:7s} c-3 tokens: median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
              f"max {L.max()} | windows of 512 needed (median) {np.median(np.ceil(L / 512)):.0f}")


# -------------------------------------------------------------------------------------
# Scorer: Micro-F1@7 (primary), Micro-P@7, Micro-R@7 as in the spec (gold not truncated or padded),
# plus ranking diagnostics that look past rank 7: Recall@100 and MRR.
#   pred: {qid: ranked list of candidate ids}, gold: {qid: set of ids}
# -------------------------------------------------------------------------------------
def evaluate_ranking(pred, gold, k=TOPK, show=True, name="dev"):
    tp = n_pred = n_gold = 0
    per_q, r100, rr = {}, [], []
    for q, g in gold.items():
        top = list(pred.get(q, []))[:k]
        hit = len(set(top) & g)
        tp += hit; n_pred += len(top); n_gold += len(g)
        per_q[q] = {"hits": hit, "gold": len(g), "f1": 2 * hit / (len(top) + len(g)) if (top or g) else 0.0}
        if g:
            full = list(pred.get(q, []))
            r100.append(len(set(full[:100]) & g) / len(g))
            first = next((i for i, c in enumerate(full) if c in g), None)
            rr.append(0.0 if first is None else 1.0 / (first + 1))
    P = tp / max(1, n_pred); R = tp / max(1, n_gold)
    F = 2 * P * R / (P + R) if P + R else 0.0
    res = {"micro_f1": F, "micro_p": P, "micro_r": R, "recall@100": float(np.mean(r100)) if r100 else 0.0,
           "mrr": float(np.mean(rr)) if rr else 0.0, "tp": tp, "per_query": per_q}
    if show:
        print(f"{name}: Micro-F1@{k} {F:.4f} | P@{k} {P:.4f} | R@{k} {R:.4f} | Recall@100 {res['recall@100']:.4f} "
              f"| MRR {res['mrr']:.4f} | queries {len(gold)}")
    return res


# -------------------------------------------------------------------------------------
# prediction.json writer: {query_id: [7 unique candidate ids from that query's test pool, best first]}
# for every test query, then submission.zip with prediction.json at the root.
# -------------------------------------------------------------------------------------
def allowed_pool(q, split):
    d = queries[split][q]
    ids = d["cands"] if d.get("cands") else pools[split].keys()
    return [c for c in ids if c != q and c in pools[split]]

def write_prediction(pred, out_dir="/kaggle/working", split="test"):
    os.makedirs(out_dir, exist_ok=True)
    out = {}
    for q in queries[split]:
        allowed = set(allowed_pool(q, split))
        top = [c for c in dict.fromkeys(pred.get(q, [])) if c in allowed][:TOPK]
        assert len(top) == TOPK, f"query {q}: only {len(top)} valid candidates"
        out[q] = top
    assert len(out) == len(queries[split])
    path = os.path.join(out_dir, "prediction.json")
    json.dump(out, open(path, "w"), indent=1)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {path} ({len(out)} queries x {TOPK}) and {zpath}")
    return out


# -------------------------------------------------------------------------------------
# BM25 over lower-cased words (k1 = 1.2, b = 0.75, the textbook defaults; not tuned).
# A lexical reference point for the report, and a source of hard negatives for training.
# -------------------------------------------------------------------------------------
_TOKRE = re.compile(r"[a-z0-9]+")

class BM25:
    def __init__(self, pool_docs, k1=1.2, b=0.75):
        self.ids = list(pool_docs)
        self.vocab = {}
        rows, cols, vals, dl = [], [], [], []
        for i, c in enumerate(tqdm(self.ids, desc="BM25 index", leave=False)):
            tf = Counter(_TOKRE.findall(pool_docs[c]["text"].lower()))
            dl.append(sum(tf.values()))
            for w, n in tf.items():
                rows.append(i); cols.append(self.vocab.setdefault(w, len(self.vocab))); vals.append(n)
        tf = sp.csr_matrix((np.array(vals, np.float32), (rows, cols)), shape=(len(self.ids), len(self.vocab)))
        dl = np.array(dl, np.float32)
        df = np.bincount(tf.indices, minlength=len(self.vocab))
        idf = np.log(1 + (len(self.ids) - df + 0.5) / (df + 0.5)).astype(np.float32)
        norm = k1 * (1 - b + b * dl / dl.mean())
        tf = tf.tocoo()
        w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col]
        self.W = sp.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape).T.tocsr()   # (V, Ndocs)

    def scores(self, text):
        q = Counter(w for w in _TOKRE.findall(text.lower()) if w in self.vocab)
        if not q:
            return np.zeros(len(self.ids), np.float32)
        qv = sp.csr_matrix((np.ones(len(q), np.float32), ([0] * len(q), [self.vocab[w] for w in q])),
                           shape=(1, len(self.vocab)))           # unique query terms (queries are whole judgments)
        return np.asarray((qv @ self.W).todense()).ravel()

def bm25_rank(split, top=200):
    """-> {qid: top `top` candidate ids of the split's pool by BM25}, self / non-allowed excluded."""
    bm = BM25(pools[split])
    pos = {c: i for i, c in enumerate(bm.ids)}
    out = {}
    for q, d in tqdm(queries[split].items(), desc=f"BM25 {split}"):
        s = bm.scores(d["text"])
        allowed = allowed_pool(q, split)
        idx = np.array([pos[c] for c in allowed])
        order = idx[np.argsort(-s[idx], kind="stable")[:top]]
        out[q] = [bm.ids[i] for i in order]
    return out

BM25_RANK = {"dev": bm25_rank("dev")}
print("\n-- BM25 baseline")
bm25_dev = evaluate_ranking(BM25_RANK["dev"], GOLD["dev"], name="BM25 dev")


8252 documents, 5819 distinct texts still to tokenise


tokenising:   0%|          | 0/5819 [00:00<?, ?it/s]

tokenised in 0.4 min, cache saved to /kaggle/working/p7_c3_tok.pkl
train queries c-3 tokens: median 6053 p90 16652 max 162313 | windows of 512 needed (median) 12
train pool    c-3 tokens: median 4813 p90 16464 max 547297 | windows of 512 needed (median) 10
dev   queries c-3 tokens: median 6168 p90 20301 max 142182 | windows of 512 needed (median) 12
dev   pool    c-3 tokens: median 5332 p90 22342 max 547297 | windows of 512 needed (median) 11
test  queries c-3 tokens: median 6232 p90 18347 max 117654 | windows of 512 needed (median) 13
test  pool    c-3 tokens: median 5224 p90 18380 max 547297 | windows of 512 needed (median) 11


BM25 index:   0%|          | 0/1023 [00:00<?, ?it/s]

BM25 dev:   0%|          | 0/118 [00:00<?, ?it/s]


-- BM25 baseline
BM25 dev: Micro-F1@7 0.1012 | P@7 0.1102 | R@7 0.0935 | Recall@100 0.4073 | MRR 0.3248 | queries 118


## Cell E0 — retrieval plumbing (c-3 bi-encoder, windows, similarity, losses, negatives, GradCache, training loop)

In [10]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared retrieval plumbing for the c-3 (hourglass) Phase 7 (PCR) runs
# Run once after Cells 0, A, B, C, D. Then run Cell E1 (the method cell).
# Needs from earlier cells: lm, cfg, device, features, queries, pools, GOLD, TRAIN_QIDS, PAD_ID,
#   evaluate_ranking, allowed_pool, write_prediction, bm25_rank, BM25_RANK
#
# Same retrieval design as the c-6 Phase 7 notebook (only the backbone differs):
# the c-3 ENCODER is used as a bi-encoder (dual encoder with shared weights):
#   case -> token windows -> ENCODER (masked) -> one vector per window (POOL)
#        -> document = its window vectors; similarity(query, candidate) from them (SIMILARITY)
#   The decoder is not used in Phase 7 (its weights stay frozen and are never run).
#   Candidates of a pool are encoded once, every query is scored against all of them.
#
# c-3 encoder (Phase 1 hourglass): embeddings -> enc_pre blocks -> hourglass (mean of every 4
#   tokens, enc_hourglass blocks on the 4x shorter sequence, repeat back x4, + residual) ->
#   enc_post blocks -> encoder_ln.
# [c-3] Padding. Phase 1 c-3 has no padding mask and only takes multiples of 4. To batch windows
#   of different lengths this cell runs the SAME blocks with the SAME weights (wrappers / 4-bit
#   layers / trained biases included, every linear and LayerNorm is called as a module) and adds
#   a key-padding mask in self-attention (full-length and bottleneck) and a masked 4-token mean
#   in the Downsample (padding never enters a group's mean; an all-padding group is masked out).
#   Every row is padded to a multiple of 4. Self-checks at the end prove this equals Phase 1.
#   (Same masked encoder as your Phase 6 c-3 notebooks.)
#
# Long documents (position table has 1536 rows, judgments are far longer): windows of WINDOW
#   tokens every STRIDE tokens (Phase 2-6 make_windows), DOC_REPR picks which ones stand for
#   the case ("all", "first", "last", "first_last", "spread"), up to MAX_WINDOWS.
#
# Similarity (choice in E1):
#   "cosine" / "dot" : window vectors -> one case vector (WIN_AGG "mean" or "max"), then cosine / dot
#   "maxsim"         : late interaction on L2-normalised window vectors: for every query window
#                      take its best-matching candidate window, average over query windows
#
# Training objective (choice in E1), one row per query with one sampled relevant candidate:
#   "infonce" : softmax cross-entropy over [its positive, its hard negatives, and (IN_BATCH_NEG)
#               every other candidate in the batch], logits = similarity / TAU
#   "margin"  : pairwise hinge  mean(max(0, MARGIN - s(q, pos) + s(q, neg))) over the same negatives
#   Candidates relevant to the query (other than the sampled positive) are never used as its
#   negatives (multi-relevant queries).
# Hard negatives (choice in E1): "none", "random" (from the pool), "bm25" (BM25 top HARD_TOP,
#   non-relevant), "model" (current model's top HARD_TOP, re-mined at the start of every epoch).
#
# Memory: CHUNK > 0 turns on gradient caching (GradCache): the batch is encoded in chunks without
#   gradients, the loss is taken on all embeddings at once, then each chunk is re-encoded with the
#   same dropout RNG and back-propagated. Same loss and gradients as one big batch.
# Nothing in this cell is a model design or a hyperparameter choice: every choice is an argument
# set in Cell E1.
# =====================================================================================
import math, time, random, os, json
import numpy as np
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model          # 560 for c-3
MAX_POS = cfg.max_seq_len       # 1536: longest encoder window
SF = cfg.shorten_factor         # [c-3] 4: hourglass shortening factor
GRAD_CHECKPOINT = False         # set in Cell E1
_AUTOCAST = torch.cuda.is_available()

def round_up(n, m=SF):
    return ((n + m - 1) // m) * m


# -------------------------------------------------------------------------------------
# Encoder with a key-padding mask (same maths as Phase 1 EncoderSelfAttention / EncoderBlock /
# hourglass; from your Phase 6 c-3 E0)
# -------------------------------------------------------------------------------------
def _heads(t, B, T, H, D):
    return t.view(B, T, H, D).transpose(1, 2)

def _scale(attn):
    return getattr(attn, "attention_scale", None) or getattr(attn, "scale", None) or 1.0 / math.sqrt(attn.d_head)

def enc_self_attn(attn, x, key_mask):
    B, T, C = x.shape
    H, D = attn.n_heads, attn.d_head
    q, k, v = [_heads(t, B, T, H, D) for t in attn.qkv_proj(x).chunk(3, dim=-1)]
    m = None if key_mask is None else key_mask[:, None, None, :]
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, scale=_scale(attn),
                                         dropout_p=attn.attn_dropout.p if attn.training else 0.0)
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

def _enc_block(blk, x, key_mask):
    x = x + enc_self_attn(blk.self_attn, blk.ln1(x), key_mask)
    return x + blk.ffn(blk.ln2(x))

def _run(backbone, fn, *args):
    ck = GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training
    return checkpoint(fn, *args, use_reentrant=False) if ck else fn(*args)

def masked_downsample(x, key_mask, f):
    """[c-3] Downsample (mean of every f tokens) that ignores padding. A group of f real tokens
    gives exactly Phase 1's Downsample. -> groups (N, T/f, C), group mask (N, T/f) or None."""
    N, T, C = x.shape
    if key_mask is None:
        return x.view(N, T // f, f, C).mean(dim=2), None
    m = key_mask.view(N, T // f, f).to(x.dtype)
    cnt = m.sum(dim=2)
    g = (x.view(N, T // f, f, C) * m[..., None]).sum(dim=2) / cnt.clamp(min=1)[..., None]
    gm = cnt > 0
    return g, (None if bool(gm.all()) else gm)

def encoder_forward(backbone, idx, key_mask=None):
    """idx (N, T) padded with PAD_ID (T a multiple of 4), key_mask (N, T) bool (True = real) -> (N, T, 560)."""
    c = backbone.core
    T = idx.size(1)
    assert T <= MAX_POS, f"encoder input of {T} tokens > {MAX_POS}"
    assert T % SF == 0, f"[c-3] encoder input length {T} is not a multiple of {SF}"
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_pre:
        x = _run(backbone, _enc_block, blk, x, key_mask)
    hg = c.encoder_hourglass
    g, gmask = masked_downsample(x, key_mask, hg.factor)
    for blk in hg.bottleneck:
        g = _run(backbone, _enc_block, blk, g, gmask)
    x = hg.upsample(g) + x
    for blk in c.encoder_post:
        x = _run(backbone, _enc_block, blk, x, key_mask)
    return c.encoder_ln(x)


# -------------------------------------------------------------------------------------
# Which ENCODER linears can be wrapped / quantized, by short name and by stage
# (the decoder is not used in Phase 7, so only enc_* names and stages exist here)
# -------------------------------------------------------------------------------------
STAGES = ("enc_pre", "enc_hourglass", "enc_post")

def stage_blocks(backbone, stage):
    c = backbone.core
    return {"enc_pre": c.encoder_pre, "enc_hourglass": c.encoder_hourglass.bottleneck,
            "enc_post": c.encoder_post}[stage]

_WHERE = {"enc_qkv": ("self_attn", "qkv_proj"), "enc_out": ("self_attn", "out_proj"),
          "enc_ffn_in": ("ffn", "net", "0"), "enc_ffn_out": ("ffn", "net", "2")}

def all_target_names(backbone=None):
    return tuple(_WHERE)

def model_linears(backbone, which, stages=STAGES):
    """Yield (parent, attr, module) for each chosen linear in every block of the chosen stages."""
    for st in stages:
        assert st in STAGES, f"unknown stage {st}; choose from {STAGES}"
    for w in which:
        assert w in _WHERE, f"unknown target {w}; choose from {tuple(_WHERE)}"
        path = _WHERE[w]
        for st in stages:
            for blk in stage_blocks(backbone, st):
                parent = blk
                for p in path[:-1]:
                    parent = getattr(parent, p)
                yield parent, path[-1], getattr(parent, path[-1])

def wrap_linears(backbone, which, make_wrapper, stages=STAGES):
    todo = list(model_linears(backbone, which, stages))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot

# -------------------------------------------------------------------------------------
# Windows (Phase 2-6 make_windows + select_windows, unchanged)
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")
POOLS = ("mean", "max", "first")
WIN_AGGS = ("mean", "max")
SIMILARITIES = ("cosine", "dot", "maxsim")
LOSSES = ("infonce", "margin")
HARD_SOURCES = ("none", "random", "bm25", "model")
POS_SAMPLINGS = ("one_per_query", "all_pairs")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]

def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)

def build_pieces(docs, window, stride, doc_repr, max_windows, desc="windows"):
    """{case_id: d} -> {case_id: [token lists, one per selected window]}."""
    assert window <= MAX_POS and stride <= window and doc_repr in DOC_REPRS
    assert doc_repr == "all" or max_windows, "set MAX_WINDOWS for this DOC_REPR"
    out = {k: [d["ids"][s:e] for s, e in select_windows(make_windows(len(d["ids"]), window, stride),
                                                           doc_repr, max_windows)]
           for k, d in tqdm(docs.items(), desc=desc, leave=False)}
    nw = np.array([len(v) for v in out.values()])
    seen = sum(sum(map(len, v)) for v in out.values()) / max(1, sum(len(d["ids"]) for d in docs.values()))
    print(f"{desc}: {len(out)} cases | windows per case mean {nw.mean():.2f} max {nw.max()} | tokens seen {seen:.1%}")
    return out


# -------------------------------------------------------------------------------------
# Bi-encoder: list of cases (each a list of windows) -> window vectors E (N, W, D) + mask M (N, W)
# -------------------------------------------------------------------------------------
def _pad(rows):
    """[c-3] rows padded to a common length rounded up to a multiple of 4 (hourglass)."""
    T = round_up(max(len(r) for r in rows))
    idx = torch.tensor([r + [PAD_ID] * (T - len(r)) for r in rows], dtype=torch.long, device=device)
    L = torch.tensor([len(r) for r in rows], device=device)
    mask = torch.arange(T, device=device)[None, :] < L[:, None]
    return idx, (None if bool(mask.all()) else mask), mask

class PCREncoder(nn.Module):
    def __init__(self, backbone, pool):
        super().__init__()
        assert pool in POOLS
        self.backbone, self.pool = backbone, pool

    def forward(self, cases):
        flat = [w for c in cases for w in c]
        idx, key_mask, tmask = _pad(flat)
        h = encoder_forward(self.backbone, idx, key_mask)            # (Nw, T, D)
        m = tmask.unsqueeze(-1).to(h.dtype)
        if self.pool == "mean":
            v = (h * m).sum(1) / m.sum(1).clamp_min(1)
        elif self.pool == "max":
            v = h.masked_fill(~tmask.unsqueeze(-1), torch.finfo(h.dtype).min).max(1).values
        else:                                                        # "first": first token of each window
            v = h[:, 0]
        W = max(len(c) for c in cases)
        E = v.new_zeros(len(cases), W, v.size(-1))
        M = torch.zeros(len(cases), W, dtype=torch.bool, device=v.device)
        i = 0
        for r, c in enumerate(cases):
            E[r, :len(c)] = v[i:i + len(c)]; M[r, :len(c)] = True
            i += len(c)
        return E, M


def similarity(qE, qM, cE, cM, sim, win_agg):
    """(Nq, Wq, D), (Nq, Wq), (Nc, Wc, D), (Nc, Wc) -> scores (Nq, Nc), fp32."""
    qE, cE = qE.float(), cE.float()
    if sim == "maxsim":
        qn, cn = F.normalize(qE, dim=-1), F.normalize(cE, dim=-1)
        s = torch.einsum("qid,cjd->qcij", qn, cn)                    # window-to-window cosine
        s = s.masked_fill(~cM[None, :, None, :], -1e4).max(-1).values  # best candidate window
        qm = qM[:, None, :].float()
        return (s * qm).sum(-1) / qm.sum(-1).clamp_min(1)            # mean over query windows
    def agg(E, M):
        if win_agg == "mean":
            m = M.unsqueeze(-1).float()
            return (E * m).sum(1) / m.sum(1).clamp_min(1)
        return E.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
    qv, cv = agg(qE, qM), agg(cE, cM)
    if sim == "cosine":
        qv, cv = F.normalize(qv, dim=-1), F.normalize(cv, dim=-1)
    return qv @ cv.T


# -------------------------------------------------------------------------------------
# Encoding a whole pool, ranking, prediction
# -------------------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(enc, pieces, ids, batch_windows, desc="encode"):
    """-> E (N, Wmax, D) fp16, M (N, Wmax) for `ids` in order. Batches by window count (less padding)."""
    enc.eval()
    order = sorted(range(len(ids)), key=lambda i: sum(map(len, pieces[ids[i]])))
    W = max(len(pieces[k]) for k in ids)
    E = torch.zeros(len(ids), W, FEAT_DIM, dtype=torch.float16, device=device)
    M = torch.zeros(len(ids), W, dtype=torch.bool, device=device)
    i, pbar = 0, tqdm(total=len(ids), desc=desc, leave=False)
    while i < len(order):
        j, nw = i, 0
        while j < len(order) and (j == i or nw + len(pieces[ids[order[j]]]) <= batch_windows):
            nw += len(pieces[ids[order[j]]]); j += 1
        sel = order[i:j]
        with torch.autocast("cuda", dtype=torch.float16, enabled=_AUTOCAST):
            e, m = enc([pieces[ids[t]] for t in sel])
        E[sel, :e.size(1)] = e.half(); M[sel, :m.size(1)] = m
        pbar.update(j - i); i = j
    pbar.close()
    return E, M

@torch.no_grad()
def rank_split(enc, split, q_pieces, c_pieces, sim, win_agg, batch_windows, top=200, q_ids=None, return_scores=False):
    """-> {qid: top `top` allowed candidate ids, best first} (and their scores if return_scores)."""
    cids = list(pools[split])
    pos = {c: i for i, c in enumerate(cids)}
    q_ids = list(queries[split]) if q_ids is None else q_ids
    cE, cM = encode_cases(enc, c_pieces, cids, batch_windows, desc=f"encode {split} pool")
    qE, qM = encode_cases(enc, q_pieces, q_ids, batch_windows, desc=f"encode {split} queries")
    out, sc = {}, {}
    qb = 8 if sim == "maxsim" else 256
    cb = 2048 if sim == "maxsim" else len(cids)
    for s in tqdm(range(0, len(q_ids), qb), desc=f"rank {split}", leave=False):
        S = torch.cat([similarity(qE[s:s + qb], qM[s:s + qb], cE[t:t + cb], cM[t:t + cb], sim, win_agg)
                       for t in range(0, len(cids), cb)], dim=1)
        for r, q in enumerate(q_ids[s:s + qb]):
            row = S[r]
            d = queries[split][q]
            if d.get("cands") or q in pos:                           # restrict to allowed candidates
                keep = torch.zeros_like(row, dtype=torch.bool)
                keep[[pos[c] for c in allowed_pool(q, split)]] = True
                row = row.masked_fill(~keep, -float("inf"))
            v, i = torch.topk(row, min(top, row.numel()))
            ok = torch.isfinite(v)
            out[q] = [cids[j] for j in i[ok].tolist()]
            sc[q] = v[ok].tolist()
    del cE, cM, qE, qM
    return (out, sc) if return_scores else out


# -------------------------------------------------------------------------------------
# Training batches
# -------------------------------------------------------------------------------------
def make_epoch_rows(pos_sampling, gold):
    """-> list of (qid, positive cid) for one epoch, shuffled."""
    rows = []
    for q in TRAIN_QIDS:
        g = sorted(gold[q])
        rows += [(q, random.choice(g))] if pos_sampling == "one_per_query" else [(q, c) for c in g]
    random.shuffle(rows)
    return rows

def pick_negatives(q, n_hard, source, hard_rank, hard_top, gold):
    if not n_hard or source == "none":
        return []
    if source == "random":
        pool_ids = TRAIN_POOL_IDS
        out = set()
        while len(out) < n_hard:
            c = random.choice(pool_ids)
            if c != q and c not in gold[q]:
                out.add(c)
        return list(out)
    cands = [c for c in hard_rank[q][:hard_top] if c not in gold[q] and c != q]
    return random.sample(cands, min(n_hard, len(cands)))

def batch_mask(rows, cand_ids, own_negs, gold, in_batch, dev):
    """rows[i] = (qid, pos); cand_ids = unique candidate ids in the batch (columns);
    own_negs[i] = set of column indices that are query i's own hard negatives.
    -> allowed (B, C) bool (positive + usable negatives), target (B,) column of each positive."""
    col = {c: j for j, c in enumerate(cand_ids)}
    B, C = len(rows), len(cand_ids)
    target = torch.tensor([col[p] for _, p in rows], device=dev)
    allowed = torch.zeros(B, C, dtype=torch.bool, device=dev)
    for i, (q, p) in enumerate(rows):
        if in_batch:
            allowed[i] = True
        else:
            allowed[i, list(own_negs[i])] = True
        for c in gold[q]:                                            # other relevant cases are not negatives
            if c in col and c != p:
                allowed[i, col[c]] = False
        if q in col:
            allowed[i, col[q]] = False
        allowed[i, col[p]] = True
    return allowed, target

def score_loss(S, allowed, target, loss_type, tau, margin):
    """Ranking loss on a (B, C) similarity matrix."""
    if loss_type == "infonce":
        return F.cross_entropy((S / tau).masked_fill(~allowed, -1e4), target)
    B = S.size(0)
    s_pos = S.gather(1, target[:, None])
    neg = allowed.clone(); neg[torch.arange(B), target] = False
    h = F.relu(margin - s_pos + S) * neg
    return (h.sum(1) / neg.sum(1).clamp_min(1)).mean()

def batch_loss(qE, qM, cE, cM, rows, cand_ids, own_negs, gold, sim, win_agg, loss_type, tau, margin, in_batch):
    S = similarity(qE, qM, cE, cM, sim, win_agg)                     # (B, C)
    allowed, target = batch_mask(rows, cand_ids, own_negs, gold, in_batch, S.device)
    return score_loss(S, allowed, target, loss_type, tau, margin)


def _rng_state():
    return (torch.get_rng_state(), torch.cuda.get_rng_state() if torch.cuda.is_available() else None)

def _set_rng(st):
    torch.set_rng_state(st[0])
    if st[1] is not None:
        torch.cuda.set_rng_state(st[1])

def step_backward(enc, q_cases, c_cases, loss_fn, chunk, scaler):
    """Forward + backward for one batch. chunk = cases per forward (None/0 = all at once).
    Returns the loss value. With chunk, GradCache: identical loss/gradients, much less memory."""
    cases = q_cases + c_cases
    nq = len(q_cases)
    ac = dict(device_type="cuda", dtype=torch.float16, enabled=_AUTOCAST)
    if not chunk or chunk >= len(cases):
        with torch.autocast(**ac):
            E, M = enc(cases)
        loss = loss_fn(E[:nq], M[:nq], E[nq:], M[nq:])
        scaler.scale(loss).backward()
        return loss.item()
    parts = [cases[i:i + chunk] for i in range(0, len(cases), chunk)]
    states, Es, Ms = [], [], []
    with torch.no_grad():
        for p in parts:
            states.append(_rng_state())
            with torch.autocast(**ac):
                e, m = enc(p)
            Es.append(e.float()); Ms.append(m)
    W = max(e.size(1) for e in Es)
    E = torch.cat([F.pad(e, (0, 0, 0, W - e.size(1))) for e in Es]).requires_grad_(True)
    M = torch.cat([F.pad(m, (0, W - m.size(1))) for m in Ms])
    loss = loss_fn(E[:nq], M[:nq], E[nq:], M[nq:])
    loss.backward()
    G = E.grad
    after = _rng_state()
    i = 0
    for p, st in zip(parts, states):
        _set_rng(st)
        with torch.autocast(**ac):
            e, _ = enc(p)
        g = G[i:i + len(p), :e.size(1)]
        scaler.scale((e.float() * g).sum()).backward()
        i += len(p)
    _set_rng(after)
    return loss.item()


# -------------------------------------------------------------------------------------
# Training loop: AdamW on trainable params (fp16 autocast + GradScaler), best epoch by dev Micro-F1@7.
# Saves best_trainable.pt and last_checkpoint.pt (resume: trainable, best, opt, scaler, sched, epoch, log).
# -------------------------------------------------------------------------------------
def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_model(enc, pieces, lr, epochs, batch_size, weight_decay, sim, win_agg, loss_type, tau, margin,
                in_batch, hard_source, n_hard, hard_top, pos_sampling, grad_clip=None, lr_lambda=None,
                chunk=None, batch_windows=64, save_path="best_trainable.pt", last_path=None,
                resume_from=None, max_epoch_minutes=None, hooks=None):
    """pieces = {"train_q", "train_c", "dev_q", "dev_c"}: case id -> windows.
    hooks (optional, e.g. the AdaLoRA allocator): extra_loss(), after_backward() (unscaled grads),
    after_step(), postfix(), budget, rank_pattern(), state_dict() / load_state_dict()."""
    params = [p for p in enc.parameters() if p.requires_grad]
    assert params, "no trainable parameters"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=_AUTOCAST)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    best, log, start = -1.0, [], 0
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        enc.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"):
            sched.load_state_dict(ck["sched"])
        best, log, start = ck["best"], ck["log"], ck["epoch"]
        if hooks is not None and ck.get("hooks") is not None:
            hooks.load_state_dict(ck["hooks"])
        torch.save(ck["best_trainable"], save_path)
        print(f"resumed after epoch {start} (best dev Micro-F1@7 {best:.4f})")
        del ck
    gold = GOLD["train"]
    hard_rank = None
    if hard_source == "bm25" and n_hard:
        if "train" not in BM25_RANK:
            BM25_RANK["train"] = bm25_rank("train", top=max(200, hard_top))
        hard_rank = BM25_RANK["train"]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start, epochs):
        if hard_source == "model" and n_hard:
            hard_rank = rank_split(enc, "train", pieces["train_q"], pieces["train_c"], sim, win_agg,
                                   batch_windows, top=hard_top + 50, q_ids=TRAIN_QIDS)
        enc.train()
        rows = make_epoch_rows(pos_sampling, gold)
        batches = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
        t0, tot, seen = time.time(), 0.0, 0
        pbar = tqdm(batches, desc=f"epoch {ep + 1}/{epochs}")
        for batch in pbar:
            negs = [pick_negatives(q, n_hard, hard_source, hard_rank, hard_top, gold) for q, _ in batch]
            cand_ids = list(dict.fromkeys([p for _, p in batch] + [c for n in negs for c in n]))
            col = {c: j for j, c in enumerate(cand_ids)}
            own = [{col[c] for c in n} for n in negs]
            q_cases = [pieces["train_q"][q] for q, _ in batch]
            c_cases = [pieces["train_c"][c] for c in cand_ids]
            loss_fn = lambda qE, qM, cE, cM: batch_loss(qE, qM, cE, cM, batch, cand_ids, own, gold, sim, win_agg,
                                                        loss_type, tau, margin, in_batch)
            opt.zero_grad(set_to_none=True)
            l = step_backward(enc, q_cases, c_cases, loss_fn, chunk, scaler)
            if hooks is not None:                                   # e.g. AdaLoRA orthogonality penalty
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=_AUTOCAST):
                    xl = hooks.extra_loss()
                if torch.is_tensor(xl) and xl.requires_grad:
                    scaler.scale(xl.float()).backward()
                    l += xl.item()
            if grad_clip or hooks is not None:
                scaler.unscale_(opt)                                 # hooks see real (unscaled) gradients
            if hooks is not None:
                hooks.after_backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched:
                sched.step()
            if hooks is not None:
                hooks.after_step()
            tot += l; seen += 1
            pf = dict(loss=f"{tot / seen:.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")
            if hooks is not None:
                pf.update(hooks.postfix())
            pbar.set_postfix(**pf)
            if max_epoch_minutes and time.time() - t0 > 60 * max_epoch_minutes:
                print(f"  time cap: stopped epoch after {seen}/{len(batches)} batches")
                break
        dev_rank = rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], sim, win_agg, batch_windows)
        r = evaluate_ranking(dev_rank, GOLD["dev"], show=False)
        row = {"epoch": ep + 1, "train_loss": tot / max(1, seen), "batches": seen,
               "minutes": (time.time() - t0) / 60, **{k: r[k] for k in ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")}}
        if hooks is not None:
            row.update({"rank_budget": hooks.budget, "ranks_kept": sum(hooks.rank_pattern())})
        log.append(row)
        improved = r["micro_f1"] > best
        if improved:
            best = r["micro_f1"]
            torch.save(trainable_state(enc), save_path)
        print(f"epoch {ep + 1}: train loss {row['train_loss']:.4f} | dev Micro-F1@7 {r['micro_f1']:.4f} "
              f"(P {r['micro_p']:.4f} R {r['micro_r']:.4f}) | R@100 {r['recall@100']:.4f} | MRR {r['mrr']:.4f}"
              f"{' *best*' if improved else ''} | {row['minutes']:.1f} min"
              + (f" | rank budget {hooks.budget}, kept {row['ranks_kept']}" if hooks is not None else ""))
        if last_path:
            torch.save({"trainable": trainable_state(enc), "best_trainable": torch.load(save_path),
                        "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": ep + 1, "best": best,
                        "log": log, "hooks": hooks.state_dict() if hooks is not None else None}, last_path)
    return log, best


# -------------------------------------------------------------------------------------
# Error breakdown helpers (numbers for the report; the analysis itself is yours)
# -------------------------------------------------------------------------------------
def breakdown(res, split="dev"):
    """Micro-F1@7 by gold-set size and by query length (c-6 tokens)."""
    pq = res["per_query"]
    def show(title, key, edges):
        print(title)
        for lo, hi in zip(edges[:-1], edges[1:]):
            qs = [q for q in pq if lo <= key(q) < hi]
            if qs:
                tp = sum(pq[q]["hits"] for q in qs); ng = sum(pq[q]["gold"] for q in qs)
                P = tp / (TOPK * len(qs)); R = tp / max(1, ng)
                print(f"  [{lo}, {hi}): {len(qs):4d} queries | Micro-F1@7 {2*P*R/(P+R) if P+R else 0:.4f}")
    show("by gold-set size:", lambda q: pq[q]["gold"], [0, 1, 3, 6, 8, 12, 10**9])
    L = {q: len(queries[split][q]["ids"]) for q in pq}
    e = [0] + [int(np.percentile(list(L.values()), p)) for p in (25, 50, 75)] + [10**9]
    show(f"by query length (quartiles, c-3 tokens):", lambda q: L[q], sorted(set(e)))

def _self_checks():
    lm.eval()
    q = next(iter(queries["dev"].values()))
    src = torch.tensor([q["ids"][:64]], device=device)
    with torch.no_grad():
        e1 = (encoder_forward(lm, src) - features(src)).abs().max().item()      # Cell B = Phase 1 path
        short = src[:, :40]
        padded = torch.cat([short, torch.full((1, 24), PAD_ID, device=device)], dim=1)
        mask = torch.arange(64, device=device)[None, :] < 40
        e2 = (encoder_forward(lm, padded, mask)[:, :40] - features(short)).abs().max().item()
        # a length that is not a multiple of 4, batched with a longer row (masked partial group)
        odd = q["ids"][:37]
        _e = PCREncoder(lm, "mean")
        E, M = _e([[odd], [q["ids"][:64]]])
        e3 = (E[1, 0] - features(src).mean(1)[0]).abs().max().item()
    print(f"self-checks max |diff|: encoder {e1:.1e} | padded encoder {e2:.1e} | batched pooled window {e3:.1e} "
          f"(all should be ~0)")
    assert max(e1, e2, e3) < 1e-3, "masked encoder differs from the Phase 1 model"

_self_checks()
TRAIN_POOL_IDS = list(pools["train"])


self-checks max |diff|: encoder 9.5e-07 | padded encoder 9.5e-07 | batched pooled window 4.8e-07 (all should be ~0)


## Cell E1 — QLoRA (fill section [H])

In [11]:
# =====================================================================================
# CELL E1 — QLoRA (Dettmers et al. 2023) on the c-3 hourglass ENCODER
#           for Phase 7 Prior Case Retrieval (bi-encoder, top-7 ranking)
#
# QLoRA = the frozen Phase 1 weights are STORED in 4 bit (bitsandbytes NF4 / FP4) and
#         de-quantized for each matmul; a full-precision LoRA adapter on top is the only thing
#         that trains:
#             y = dequant(W_4bit) x + (alpha / r) B (A x) + b
# The adapter is your own LoRALinear (Phase 2 c-6 / c-3 cell, same as your Phase 3 and Phase 6
# c-3 QLoRA notebooks), unchanged. This is additive LoRA on a 4-bit base, NOT LORMA.
#
# What gets quantized: the encoder nn.Linear layers named in QUANT_TARGETS inside the blocks of
# QUANT_STAGES. Token/position embeddings and LayerNorms stay full precision. The decoder is not
# used in Phase 7 and is left as it is. A linear can be quantized and LoRA-wrapped, only
# quantized, or only LoRA-wrapped.
#
# Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# (this cell replaces linears in place with 4-bit ones; it refuses to quantize twice).
#
# >>> FILL IN: every value marked  # <<< FILL  in section [H]. Nothing else needs changing. <<<
# Sections marked [ANCILLARY] are plumbing.
# =====================================================================================
try:
    import bitsandbytes as bnb
except ImportError:                       # Kaggle image without bitsandbytes: install once
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
    import bitsandbytes as bnb
print("bitsandbytes", bnb.__version__)

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — YOURS (all left unset on purpose)
# -------------------------------------------------------------------------------------
# long case -> windows (same for queries and candidates)
WINDOW        = 512
STRIDE        = 256
DOC_REPR      = "spread"
MAX_WINDOWS   = 8

POOL          = "mean"
SIMILARITY    = "cosine"
WIN_AGG       = "mean"

LOSS          = "infonce"
TAU           = 0.05
MARGIN        = 0.2
IN_BATCH_NEG  = True
HARD_NEG      = "bm25"
N_HARD        = 4
HARD_TOP      = 50
POS_SAMPLING  = "one_per_query"

LORA_RANK     = 80                                                    # = your Phase 7 LORMA_RANK
LORA_ALPHA    = 80                                                    # alpha/r = 1 (see below)
LORA_TARGETS  = ("enc_qkv", "enc_out", "enc_ffn_in", "enc_ffn_out")   # = your LORMA_TARGETS
LORA_STAGES   = ("enc_pre", "enc_hourglass", "enc_post")

QUANT_TARGETS = ("enc_qkv", "enc_out", "enc_ffn_in", "enc_ffn_out")   # Phase 3: every encoder linear
QUANT_STAGES  = ("enc_pre", "enc_hourglass", "enc_post")
QUANT_TYPE    = "nf4"
DOUBLE_QUANT  = True
COMPUTE_DTYPE = torch.float16

LR            = 3e-4      # c-6 LORMA (your c-3 BitFit used 1e-4)
EPOCHS        = 6
BATCH_SIZE    = 16
WEIGHT_DECAY  = 0.01      # c-6 LORMA (BitFit used 0.0)
GRAD_CLIP     = 1.0
LR_LAMBDA     = None
MAX_EPOCH_MINUTES = None
SEED          = 42

GRAD_CHECKPOINT = True
CHUNK           = 4
ENC_BATCH_WINDOWS = 64     # windows per forward when encoding pools for ranking (lower it on OOM)
# [ANCILLARY] continue an earlier run: path to its last_checkpoint.pt (attach that notebook's output as input;
# find it with  !find /kaggle/input -name last_checkpoint.pt ). None = fresh start. Keep every [H] value the same.
RESUME_FROM     = None
# [ANCILLARY] save the final model's rankings + scores (train/dev/test, top 100) for a KD notebook later
DUMP_TEACHER    = True

_required = ["WINDOW", "STRIDE", "DOC_REPR", "POOL", "SIMILARITY", "LOSS", "IN_BATCH_NEG", "HARD_NEG",
             "N_HARD", "POS_SAMPLING", "LORA_RANK", "LORA_ALPHA", "LORA_TARGETS", "LORA_STAGES",
             "QUANT_TARGETS", "QUANT_STAGES", "QUANT_TYPE", "DOUBLE_QUANT", "COMPUTE_DTYPE", "LR", "EPOCHS", "BATCH_SIZE",
             "WEIGHT_DECAY", "GRAD_CLIP", "SEED"]
_required += ["TAU"] if LOSS == "infonce" else ["MARGIN"] if LOSS == "margin" else []
_required += ["WIN_AGG"] if SIMILARITY in ("cosine", "dot") else []
_required += ["HARD_TOP"] if HARD_NEG in ("bm25", "model") and N_HARD else []
_required += ["MAX_WINDOWS"] if DOC_REPR not in (None, "all") else []
_unset = [n for n in _required if globals()[n] is None]

# [ANCILLARY] which encoder linears exist (sizes from the loaded model; every block has the same linears)
print("target names for LORA_TARGETS / QUANT_TARGETS (c-3 encoder):")
for _name in all_target_names(lm):
    _parent, _attr, _lin = next(model_linears(lm, (_name,)))
    print(f"  {_name:12s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")
print("blocks per stage:", {s: len(stage_blocks(lm, s)) for s in STAGES},
      "| enc_hourglass runs on the 4x shorter sequence")

assert not _unset, f"fill in section [H] first: {_unset}"
assert STRIDE <= WINDOW <= cfg.max_seq_len and DOC_REPR in DOC_REPRS
assert POOL in POOLS and SIMILARITY in SIMILARITIES and LOSS in LOSSES
assert HARD_NEG in HARD_SOURCES and POS_SAMPLING in POS_SAMPLINGS
assert SIMILARITY == "maxsim" or WIN_AGG in WIN_AGGS
assert IN_BATCH_NEG or (N_HARD and HARD_NEG != "none"), "no negatives at all: turn on IN_BATCH_NEG or hard negatives"
for _n in ("LORA_TARGETS", "LORA_STAGES", "QUANT_TARGETS", "QUANT_STAGES"):
    assert isinstance(globals()[_n], (tuple, list)) and globals()[_n], f"{_n} must be a non-empty tuple"
for _n in ("LORA_STAGES", "QUANT_STAGES"):
    assert all(s in STAGES for s in globals()[_n]), f"{_n}: choose from {STAGES}"
assert QUANT_TYPE in ("nf4", "fp4"), 'QUANT_TYPE: "nf4" or "fp4"'
assert isinstance(DOUBLE_QUANT, bool), "DOUBLE_QUANT: True or False"
assert COMPUTE_DTYPE in (torch.float16, torch.float32, torch.bfloat16), "COMPUTE_DTYPE: a torch dtype"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
RANKING = dict(sim=SIMILARITY, win_agg=WIN_AGG)
OBJ = dict(loss_type=LOSS, tau=TAU, margin=MARGIN, in_batch=IN_BATCH_NEG, hard_source=HARD_NEG,
           n_hard=N_HARD, hard_top=HARD_TOP, pos_sampling=POS_SAMPLING)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c3_qlora_pcr_{SIMILARITY}_{LOSS}_{HARD_NEG}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
LAST = f"{OUT_DIR}/last_checkpoint.pt"
QUANT = dict(targets=list(QUANT_TARGETS), stages=list(QUANT_STAGES), quant_type=QUANT_TYPE,
             double_quant=DOUBLE_QUANT, compute_dtype=str(COMPUTE_DTYPE))


# -------------------------------------------------------------------------------------
# [QLoRA] frozen nn.Linear -> bitsandbytes 4-bit linear (same weights, quantized). [ANCILLARY]
# -------------------------------------------------------------------------------------
def to_4bit(lin):
    assert type(lin) is nn.Linear, "already quantized or wrapped: re-run Cell A and E0"
    q = bnb.nn.Linear4bit(lin.in_features, lin.out_features, bias=lin.bias is not None,
                          compute_dtype=COMPUTE_DTYPE, compress_statistics=DOUBLE_QUANT, quant_type=QUANT_TYPE)
    q.weight = bnb.nn.Params4bit(lin.weight.data.detach().float().cpu(), requires_grad=False,
                                 quant_type=QUANT_TYPE, compress_statistics=DOUBLE_QUANT)
    if lin.bias is not None:
        q.bias = nn.Parameter(lin.bias.data.detach().clone().cpu(), requires_grad=False)
    q = q.to(device)            # moving the weight to the device is the step that quantizes it
    assert getattr(q.weight, "quant_state", None) is not None, "bitsandbytes did not quantize the weight"
    return q

def _weight_mb(which, stages):
    return sum(l.weight.numel() * l.weight.element_size() for _, _, l in model_linears(lm, which, stages)) / 2**20


# -------------------------------------------------------------------------------------
# [L] LoRA layer — yours (Phase 2 c-6 / c-3 LoRALinear, unchanged): W' = W + (alpha / r) B A
#   Here `base` is the frozen 4-bit layer when the linear is also in QUANT_TARGETS / QUANT_STAGES.
#   Note from your Phase 2 c-6 run: A ~ N(0, 1) was stable with ffn_out only, but went NaN
#   under fp16 when all four encoder targets were wrapped. If you wrap many linears, revisit
#   the init scale of A (GradScaler skips steps whose gradients are inf / NaN).
# -------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.rank, self.alpha = rank, alpha
        self.A = nn.Linear(self.in_features, rank, bias=False)
        self.B = nn.Linear(rank, self.out_features, bias=False)

        nn.init.normal_(self.A.weight, mean=0.0, std=1.0)
        nn.init.zeros_(self.B.weight)

        self.scaling = alpha / rank
    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))


def lora_on(base):
    """[ANCILLARY] wrap a (4-bit or full-precision) frozen linear; only A and B are moved to the device,
    because calling .to() again on an already-quantized bitsandbytes layer is not safe."""
    w = LoRALinear(base, LORA_RANK, LORA_ALPHA)
    w.A.to(device); w.B.to(device)
    return w


# [ANCILLARY] quick checks on a dummy linear: shape, and that it really is additive LoRA
_t = LoRALinear(nn.Linear(560, 1680, bias=True), rank=LORA_RANK, alpha=LORA_ALPHA)
_x = torch.randn(2, 7, 560)
assert _t(_x).shape == (2, 7, 1680)
with torch.no_grad():
    nn.init.normal_(_t.B.weight, std=0.02)
    _merged = _t.base.weight + (LORA_ALPHA / LORA_RANK) * (_t.B.weight @ _t.A.weight)   # W + (alpha/r) B A
    assert torch.allclose(_t(_x), _x @ _merged.T + _t.base.bias, atol=1e-3, rtol=1e-4), "layer is not W + (alpha/r) BA"
print("LoRALinear = W + (alpha/r) BA checked | trainable params on a 560->1680 layer:",
      sum(p.numel() for p in _t.parameters() if p.requires_grad))
del _t, _x, _merged


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze c-3, quantize, wrap with LoRA, check: quantization error, identity at init,
#             only A/B train
# -------------------------------------------------------------------------------------
GRAD_CHECKPOINT = GRAD_CHECKPOINT       # read by encoder_forward in E0
lm.eval()
_probe = torch.tensor([next(iter(queries["dev"].values()))["ids"][:64]], device=device)
def _probe_out():
    with torch.no_grad():
        return encoder_forward(lm, _probe).float()

for p in lm.parameters():
    p.requires_grad = False
n_params_fp = sum(p.numel() for p in lm.parameters())
ref_fp = _probe_out()

mb_fp = _weight_mb(QUANT_TARGETS, QUANT_STAGES)
n_quant = wrap_linears(lm, QUANT_TARGETS, to_4bit, stages=QUANT_STAGES)
assert n_quant > 0, "nothing quantized"
if torch.cuda.is_available():
    torch.cuda.empty_cache()
mb_q = _weight_mb(QUANT_TARGETS, QUANT_STAGES)
ref_q = _probe_out()
rel = ((ref_q - ref_fp).norm() / ref_fp.norm()).item()
cos_q = F.cosine_similarity(ref_q.mean(1), ref_fp.mean(1)).item()
print(f"[QLoRA] {n_quant} encoder linears ({QUANT_TARGETS} in {QUANT_STAGES}) -> 4-bit {QUANT_TYPE}, double quant "
      f"{DOUBLE_QUANT}, compute {COMPUTE_DTYPE}\n        their weights {mb_fp:.1f} MB -> {mb_q:.1f} MB "
      f"(+ per-block scales) | probe encoder output change from quantization: relative {rel:.3e} | "
      f"cosine of mean-pooled vectors fp vs 4-bit {cos_q:.4f}")

n_wrapped = wrap_linears(lm, LORA_TARGETS, lora_on, stages=LORA_STAGES)
assert n_wrapped > 0, "no linear matched LORA_TARGETS / LORA_STAGES"
_n_on_4bit = sum(isinstance(m.base, bnb.nn.Linear4bit) for _, _, m in model_linears(lm, LORA_TARGETS, LORA_STAGES))
print(f"wrapped {n_wrapped} linears with LoRA ({LORA_TARGETS} in {LORA_STAGES}); "
      f"{_n_on_4bit} of them sit on a 4-bit base, {n_wrapped - _n_on_4bit} on a full-precision base")
assert all(isinstance(m, LoRALinear) for _, _, m in model_linears(lm, LORA_TARGETS, LORA_STAGES))

diff = (_probe_out() - ref_q).abs().max().item()
print(f"QLoRA identity-at-init max |diff| vs the 4-bit base = {diff:.2e} (should be ~0)")
assert diff < 1e-3, "wrapped model differs from the 4-bit base at init"

enc = PCREncoder(lm, POOL)        # [QLoRA] no .to(device): everything is already placed, and .to() on 4-bit layers is unsafe
_lora_ids = {id(p) for m in enc.modules() if isinstance(m, LoRALinear) for p in (m.A.weight, m.B.weight)}
_other = [n for n, p in enc.named_parameters() if p.requires_grad and id(p) not in _lora_ids]
assert not _other, f"only LoRA parameters should train, found {_other[:3]}"
_lora_n = sum(p.numel() for p in enc.parameters() if p.requires_grad)
print(f"trainable {_lora_n/1e6:.3f}M LoRA A/B values | Phase 1 c-3 has {n_params_fp/1e6:.2f}M parameters "
      f"({100*_lora_n/n_params_fp:.2f}%)  (count_params below sees 4-bit weights as packed bytes)")
count_params(enc)

CFG_OUT = {"run": RUN_NAME, "method": "QLoRA: 4-bit base + LoRA W+(alpha/r)BA, c-3 encoder", "build": BUILD,
           "pool": POOL, "ranking": RANKING, "objective": OBJ, "quant": QUANT, "quant_probe_rel_err": rel,
           "quant_weight_mb": [mb_fp, mb_q], "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
           "lora_targets": list(LORA_TARGETS), "lora_stages": list(LORA_STAGES), "trainable": _lora_n}


# -------------------------------------------------------------------------------------
# [ANCILLARY] windows for every query and candidate
# -------------------------------------------------------------------------------------
pieces = {"train_q": build_pieces({q: queries["train"][q] for q in TRAIN_QIDS}, **BUILD, desc="train queries"),
          "train_c": build_pieces(pools["train"], **BUILD, desc="train pool"),
          "dev_q": build_pieces(queries["dev"], **BUILD, desc="dev queries"),
          "dev_c": build_pieces(pools["dev"], **BUILD, desc="dev pool"),
          "test_q": build_pieces(queries["test"], **BUILD, desc="test queries"),
          "test_c": build_pieces(pools["test"], **BUILD, desc="test pool")}


# -------------------------------------------------------------------------------------
# [ANCILLARY] GradCache self-check: chunked gradients == one-pass gradients (eval mode, tiny batch)
# -------------------------------------------------------------------------------------
if CHUNK:
    _rows = [(q, sorted(GOLD["train"][q])[0]) for q in TRAIN_QIDS[:3]]
    _cids = list(dict.fromkeys([p for _, p in _rows] + TRAIN_POOL_IDS[:3]))
    _short = lambda ws: [w[:32] for w in ws[:2]]
    _qc = [_short(pieces["train_q"][q]) for q, _ in _rows]
    _cc = [_short(pieces["train_c"][c]) for c in _cids]
    _own = [set() for _ in _rows]
    _lf = lambda qE, qM, cE, cM: batch_loss(qE, qM, cE, cM, _rows, _cids, _own, GOLD["train"], SIMILARITY,
                                            WIN_AGG, LOSS, TAU, MARGIN, True)
    _params = [p for p in enc.parameters() if p.requires_grad]
    _zero_init = [m.B.weight for m in enc.modules() if isinstance(m, LoRALinear)]          # adapter tensors that start at 0 (made non-zero for the check, then reset)
    with torch.no_grad():
        for p in _zero_init:
            p.normal_(0, 0.02)
    _sc = torch.cuda.amp.GradScaler(enabled=False)
    enc.eval()
    _AUTOCAST = False        # compare in fp32: without loss scaling, fp16 bias/adapter gradients underflow in
                             # this tiny check and the two paths lose different amounts (training uses GradScaler)
    grads = []
    for ch in (None, 2):
        for p in _params:
            p.grad = None
        step_backward(enc, _qc, _cc, _lf, ch, _sc)
        grads.append(torch.cat([p.grad.flatten().float() for p in _params]))
    _rel = ((grads[0] - grads[1]).norm() / grads[0].norm().clamp_min(1e-12)).item()
    _AUTOCAST = torch.cuda.is_available()
    print(f"GradCache check (fp32): relative gradient difference {_rel:.2e} (should be ~1e-5 or less)")
    assert _rel < 5e-2, "chunked gradients differ from one-pass gradients"
    with torch.no_grad():
        for p in _zero_init:
            p.zero_()
        for p in _params:
            p.grad = None
    del grads, _params, _zero_init


# -------------------------------------------------------------------------------------
# [ANCILLARY] zero-shot Phase 1 model (LoRA B = 0: the 4-bit quantized c-3 encoder) on dev, for the report
# -------------------------------------------------------------------------------------
print("\n-- dev before training (Phase 1 c-3 encoder as-is, quantized to 4 bit)")
zero_shot = evaluate_ranking(rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING,
                                        batch_windows=ENC_BATCH_WINDOWS), GOLD["dev"], name="zero-shot dev")
print(f"BM25 dev Micro-F1@7 {bm25_dev['micro_f1']:.4f}")


# -------------------------------------------------------------------------------------
# [ANCILLARY] training, dev scoring, test submission
# -------------------------------------------------------------------------------------
t_train = time.time()
log, best = train_model(enc, pieces, lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
                        **RANKING, **OBJ, grad_clip=GRAD_CLIP, lr_lambda=LR_LAMBDA, chunk=CHUNK,
                        batch_windows=ENC_BATCH_WINDOWS, save_path=SAVE, last_path=LAST,
                        resume_from=RESUME_FROM, max_epoch_minutes=MAX_EPOCH_MINUTES)
train_min = (time.time() - t_train) / 60
train_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0

enc.load_state_dict(torch.load(SAVE), strict=False)

t0 = time.time()
dev_rank, dev_scores = rank_split(enc, "dev", pieces["dev_q"], pieces["dev_c"], **RANKING,
                                  batch_windows=ENC_BATCH_WINDOWS, top=100, return_scores=True)
dev_s = time.time() - t0
print("\n-- dev (best epoch)")
res = evaluate_ranking(dev_rank, GOLD["dev"], name="QLoRA dev")
breakdown(res)
print(f"zero-shot dev Micro-F1@7 {zero_shot['micro_f1']:.4f} | BM25 {bm25_dev['micro_f1']:.4f} | "
      f"QLoRA {res['micro_f1']:.4f}")
print(f"training {train_min:.1f} min | dev ranking {dev_s:.0f} s ({len(dev_rank)} queries x {len(pools['dev'])} "
      f"candidates) | peak training memory {train_mem:.2f} GB | quantized weights {mb_fp:.1f} -> {mb_q:.1f} MB")
_metrics = ("micro_f1", "micro_p", "micro_r", "recall@100", "mrr")
json.dump({**CFG_OUT, "log": log, "train_minutes": train_min, "peak_mem_gb": train_mem, "dev_rank_seconds": dev_s,
           "dev": {k: res[k] for k in _metrics},
           "zero_shot_dev": {k: zero_shot[k] for k in _metrics},
           "bm25_dev": {k: bm25_dev[k] for k in _metrics},
           "dev_rank": {q: v[:20] for q, v in dev_rank.items()}}, open(f"{OUT_DIR}/val_log.json", "w"))

# Test ranking with the best weights -> /kaggle/working/<RUN_NAME>/submission.zip
test_rank, test_scores = rank_split(enc, "test", pieces["test_q"], pieces["test_c"], **RANKING,
                                    batch_windows=ENC_BATCH_WINDOWS, top=100, return_scores=True)
write_prediction(test_rank, out_dir=OUT_DIR)

# [ANCILLARY] everything a KD notebook needs: rankings + similarity scores, and how to rebuild this model
if DUMP_TEACHER:
    tr_rank, tr_scores = rank_split(enc, "train", pieces["train_q"], pieces["train_c"], **RANKING,
                                    batch_windows=ENC_BATCH_WINDOWS, top=100, q_ids=TRAIN_QIDS, return_scores=True)
    json.dump({s: {q: {"ids": r[q], "scores": [round(x, 5) for x in sc[q]]} for q in r}
               for s, r, sc in (("train", tr_rank, tr_scores), ("dev", dev_rank, dev_scores),
                                ("test", test_rank, test_scores))},
              open(f"{OUT_DIR}/teacher_rankings.json", "w"))
json.dump({**CFG_OUT, "weights": "best_trainable.pt"}, open(f"{OUT_DIR}/teacher_config.json", "w"), indent=1)
print("files for later runs / KD:", sorted(os.listdir(OUT_DIR)))


bitsandbytes 0.50.2
target names for LORA_TARGETS / QUANT_TARGETS (c-3 encoder):
  enc_qkv        560 ->  1680  bias=False
  enc_out        560 ->   560  bias=False
  enc_ffn_in     560 ->  1000  bias=True
  enc_ffn_out   1000 ->   560  bias=True
blocks per stage: {'enc_pre': 2, 'enc_hourglass': 4, 'enc_post': 2} | enc_hourglass runs on the 4x shorter sequence
LoRALinear = W + (alpha/r) BA checked | trainable params on a 560->1680 layer: 179200


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (560) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (1000) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[QLoRA] 32 encoder linears (('enc_qkv', 'enc_out', 'enc_ffn_in', 'enc_ffn_out') in ('enc_pre', 'enc_hourglass', 'enc_post')) -> 4-bit nf4, double quant True, compute torch.float16
        their weights 72.5 MB -> 9.1 MB (+ per-block scales) | probe encoder output change from quantization: relative 2.007e-02 | cosine of mean-pooled vectors fp vs 4-bit 0.9998
wrapped 32 linears with LoRA (('enc_qkv', 'enc_out', 'enc_ffn_in', 'enc_ffn_out') in ('enc_pre', 'enc_hourglass', 'enc_post')); 32 of them sit on a 4-bit base, 0 on a full-precision base
QLoRA identity-at-init max |diff| vs the 4-bit base = 0.00e+00 (should be ~0)
trainable 4.147M LoRA A/B values | Phase 1 c-3 has 55.13M parameters (7.52%)  (count_params below sees 4-bit weights as packed bytes)
trainable 4.147M / total 49.78M (8.33%)


train queries:   0%|          | 0/817 [00:00<?, ?it/s]

train queries: 817 cases | windows per case mean 7.68 max 8 | tokens seen 42.6%


train pool:   0%|          | 0/4320 [00:00<?, ?it/s]

... (56 outputs left out)

encode train pool:   0%|          | 0/4320 [00:00<?, ?it/s]

encode train queries:   0%|          | 0/817 [00:00<?, ?it/s]

rank train:   0%|          | 0/4 [00:00<?, ?it/s]

files for later runs / KD: ['best_trainable.pt', 'last_checkpoint.pt', 'prediction.json', 'submission.zip', 'teacher_config.json', 'teacher_rankings.json', 'val_log.json']
